In [ ]:
!pip install transformers datasets torch accelerate

Se agrega al inicio el import de los datasets:
- SOLIDIFY BENCHMARK
- SLITHER AUDITED SMART CONTRACTS DATASET
Se agrega una función para obtener solo el último contrato del código, sin incluir el pragma.

In [1]:
import ast
import os
import re
import sys
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from datasets import Dataset as HFDataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    RobertaTokenizer,
    Trainer,
    TrainingArguments,
)

In [ ]:
###########################################
### CREACIÓN DATASET SOLIDIFY-BENCHMARK ###
###########################################


def build_solidify_dataset(base_path):
    # Estructura: { codigo_fuente: set([cat1, cat2, ...]) }
    # Usamos un set para que si el mismo bug es reportado por 5 herramientas,
    # solo guarde una vez la etiqueta.
    raw_data = {}

    herramientas = ["Manticore", "Mythril", "Oyente", "Securify", "Slither", "SmartCheck"]

    carpetas_vulnerabilidades = [
        "Overflow-Underflow",
        "Re-entrancy",
        "TOD",
        "Timestamp-Dependency",
        "Unchecked-Send",
        "Unhandled-Exceptions",
        "tx.origin",
    ]

    print("Iniciando escaneo de directorios de SolidiFI...")

    for tool in herramientas:
        tool_path = os.path.join(base_path, tool, "analyzed_buggy_contracts")

        if not os.path.exists(tool_path):
            continue

        for folder_name in carpetas_vulnerabilidades:
            cat_path = os.path.join(tool_path, folder_name)
            if not os.path.exists(cat_path):
                continue

            for file_name in os.listdir(cat_path):
                if file_name.endswith(".sol"):
                    sol_path = os.path.join(cat_path, file_name)

                    try:
                        with open(sol_path, "r", encoding="utf-8", errors="ignore") as f:
                            code = f.read().strip()
                    except:
                        continue

                    if not code:
                        continue

                    if code not in raw_data:
                        raw_data[code] = set()

                    # Usamos folder_name directamente como etiqueta
                    raw_data[code].add(folder_name)

    # Convertimos a DataFrame
    final_list = []
    for code, categories_set in raw_data.items():
        final_list.append(
            {
                "source_code": code,
                "categories": list(categories_set),  # Guardamos como lista [cat1, cat2]
            }
        )

    df_final = pd.DataFrame(final_list)
    return df_final


# --- EJECUCIÓN ---
path = "/kaggle/input/datasets/frmorales/solidify-benchmark/results"
df_solidifi = build_solidify_dataset(path)

# Verificación de multietiqueta
multilabel_contracts = df_solidifi[df_solidifi["categories"].apply(len) > 1]

print(f"📊 Total contratos únicos: {len(df_solidifi)}")
print(f"⚠️ Contratos con más de una vulnerabilidad: {len(multilabel_contracts)}")
print("-" * 30)
print(df_solidifi.head())

In [ ]:
#################################################
### ALMACENAMIENTO DATASET SOLIDIFY-BENCHMARK ###
#################################################


def save_clean_summary(df, filename="solidify_mapping.csv"):
    """
    Guarda un resumen del mapeo de vulnerabilidades por contrato.
    Estructura: Entrada ID | Categorías (separadas por comas) | Código (truncado para vista previa)
    """
    # 1. Creamos la copia para no afectar el df original de memoria
    df_export = df.copy()

    # 2. Generamos el ID de entrada (Entrada 1, Entrada 2...)
    df_export["entrada"] = [f"Entrada {i + 1}" for i in range(len(df))]

    # 3. Formateamos la columna 'categories' para que sea legible en CSV
    # De: ['arithmetic', 'reentrancy'] -> A: "arithmetic, reentrancy"
    df_export["vulnerabilidades"] = df_export["categories"].apply(lambda x: ", ".join(x))

    # 4. Agregamos una vista previa del código (primeros 50 caracteres)
    # solo para que el CSV sea fácil de inspeccionar visualmente
    df_export["preview_codigo"] = df_export["source_code"].apply(lambda x: x[:50].replace("\n", " ") + "...")

    # 5. Seleccionamos y ordenamos las columnas para el CSV final
    df_final_export = df_export[["entrada", "vulnerabilidades", "preview_codigo"]]

    # 6. Guardamos
    df_final_export.to_csv(filename, index=False, encoding="utf-8")

    print(f"✅ CSV legible guardado como: {filename}")
    print(f"📊 Total de filas exportadas: {len(df_final_export)}")


# --- EJECUCIÓN ---
save_clean_summary(df_solidifi)

In [ ]:
###############################################
### ESTADÍSTICAS DATASET SOLIDIFY BENCHMARK ###
###############################################

# 1. Información General
total_contratos = len(df_solidifi)
df_solidifi["source_len"] = df_solidifi["source_code"].str.len()

print(f"📊 Cantidad total de contratos únicos: {total_contratos}")
print(f"📏 Longitud promedio del código (caracteres): {df_solidifi['source_len'].mean():.2f}")
print("-" * 50)

# 2. Conteo de Vulnerabilidades (Total)
# Como 'categories' es una lista, la "aplanamos" para contar
todas_las_vulns = [cat for lista in df_solidifi["categories"] for cat in lista]
conteo_vulns = Counter(todas_las_vulns)
df_vulns = pd.Series(conteo_vulns).sort_values(ascending=False)

print("### DISTRIBUCIÓN DE VULNERABILIDADES (Ground Truth) ###")
print(df_vulns)
print("-" * 50)

In [3]:
###############################################
### SLITHER AUDITED SMART CONTRACTS DATASET ###
###############################################
import glob

import pandas as pd

# Reemplaza 'nombre-del-dataset' con el nombre que aparece en la salida del paso anterior
dataset_path = "/kaggle/input/datasets/grawatschp/slither-audited-smart-contracts"

# Buscamos todos los archivos .parquet dentro de esa carpeta
parquet_files = glob.glob(f"{dataset_path}/**/*.parquet", recursive=True)

print(f"Se encontraron {len(parquet_files)} archivos Parquet.")

# Cargamos y concatenamos todos los archivos
df_list = [pd.read_parquet(file) for file in parquet_files]
df_slither = pd.concat(df_list, ignore_index=True)

print(f"✅ Dataset cargado con éxito. Total de filas: {len(df_slither)}")

# Visualización rápida de las primeras filas
display(df_slither.head())

Se encontraron 3 archivos Parquet.
✅ Dataset cargado con éxito. Total de filas: 14134


,address,source_code,bytecode,slither
0,0x01b23286ff60a543ec29366ae8d6b6274ca20541,pragma solidity 0.4.26;\n\ninterface IERC20 {\...,0x608060405260043610610112576000357c0100000000...,[6]
1,0x0cfb151de2c34aceb532f43683e5b7bed62f298f,pragma solidity 0.6.12;\npragma experimental A...,0x608060405260043610620002475760003560e01c8063...,"[5, 2, 6, 1, 0, 3]"
2,0x0e68432827674ad048b803d1ee289ae78b3917b9,pragma solidity 0.6.12;\npragma experimental A...,0x6080604052600436106101185760003560e01c80638d...,"[5, 2, 6, 1, 0, 3]"
3,0x1149d772bce9a636d0d7535ec865f3c6c8ee3b5c,pragma solidity 0.6.10;\npragma experimental A...,0x608060405234801561001057600080fd5b5060043610...,"[1, 3, 7, 2]"
4,0x11c26446b5ce3b895ef6a9a594cf9df6e8badbd7,pragma solidity 0.6.5;\npragma experimental AB...,0x608060405234801561001057600080fd5b5060043610...,[4]


In [6]:
###############################################
### SLITHER AUDITED SMART CONTRACTS DATASET ###
###############################################

from collections import Counter

import pandas as pd

# 1. Mapeo Oficial del dataset Slither Audited
# ID 4 es 'safe' (limpio). Los demás son vulnerabilidades.
mapping = {
    0: "access-control",
    1: "arithmetic",
    2: "other",
    3: "reentrancy",
    4: "safe",
    5: "unchecked-calls",
    6: "constant-outcome",
    7: "access-control",
}


def analyze_slither_labels_refined(df):
    all_vulnerabilities = []
    clean_contracts = 0

    for label_list in df["slither"]:
        # Manejo de arrays de NumPy para evitar el ValueError
        # Si es None, está vacío o el primer elemento es 4 (safe) sin otros bugs
        if label_list is None or len(label_list) == 0:
            clean_contracts += 1
            continue

        # Convertimos a set para eliminar duplicados en el mismo contrato
        unique_ids = set(label_list)

        # Caso especial: Si solo tiene el ID 4, es un contrato limpio
        if unique_ids == {4}:
            clean_contracts += 1
        else:
            # Filtramos el 4 y mapeamos los demás
            bugs_found = [mapping.get(i, f"unknown({i})") for i in unique_ids if i != 4]
            if not bugs_found:  # Por si venía algo raro
                clean_contracts += 1
            else:
                all_vulnerabilities.extend(bugs_found)

    counts = Counter(all_vulnerabilities)
    counts["clean"] = clean_contracts
    return counts


# --- EJECUCIÓN DEL ANÁLISIS ---

print(f"📊 Cantidad total de contratos: {len(df_slither)}")

# Longitud del código
df_slither["source_len"] = df_slither["source_code"].str.len()
print(f"📏 Longitud promedio del código (caracteres): {df_slither['source_len'].mean():.2f}")
print(f"🔝 Longitud máxima: {df_slither['source_len'].max()}")

# Distribución
conteo_vulnerabilidades = analyze_slither_labels_refined(df_slither)

print("\n### DISTRIBUCIÓN DE VULNERABILIDADES (Mapeado) ###")
# Ordenamos por los más comunes
for label, count in conteo_vulnerabilidades.most_common():
    print(f"{label.ljust(25)} \t {count}")


# Análisis multietiqueta (Bugs reales, excluyendo el ID 4)
def count_real_bugs(x):
    if x is None:
        return 0
    real_bugs = set(x) - {4}
    return len(real_bugs)


multilabel_df = df_slither["slither"].apply(count_real_bugs)
print(f"\n⚠️ Contratos con más de una vulnerabilidad distinta: {(multilabel_df > 1).sum()}")
print(f"✅ Contratos totalmente limpios (ID 4 o vacío): {conteo_vulnerabilidades['clean']}")

📊 Cantidad total de contratos: 14134
📏 Longitud promedio del código (caracteres): 29986.09
🔝 Longitud máxima: 721943

### DISTRIBUCIÓN DE VULNERABILIDADES (Mapeado) ###
constant-outcome          	 7768
access-control            	 4501
other                     	 4098
unchecked-calls           	 3480
clean                     	 3000
reentrancy                	 2426
arithmetic                	 1954
unknown(8)                	 501

⚠️ Contratos con más de una vulnerabilidad distinta: 5986
✅ Contratos totalmente limpios (ID 4 o vacío): 3000


In [2]:
#####################################
### IMPORT DATASET SMARTBUGS WILD ###
#####################################

df_wild = pd.read_csv("/kaggle/input/datasets/tranduongminhdai/smartbug-dataset/smartbugs_wild.csv")

In [3]:
###########################################
### ESTADÍSTICAS DATASET SMARTBUGS WILD ###
###########################################

print(f"Cantidad total de entradas: {len(df_wild)}")

df_wild["source_len"] = df_wild["source_code"].str.len()
print(f"Longitud promedio del código (caracteres): {df_wild['source_len'].mean():.2f}")

Cantidad total de entradas: 47451
Longitud promedio del código (caracteres): 14019.29


In [4]:
def extract_labels(tools_dict):
    labels = set()
    for tool, result in tools_dict.items():
        # Extraigo categories porque es un problema más sencillo que las vulnerabilidades en sí
        for vuln in result.get("categories", {}).keys():
            labels.add(vuln.lower())
    return labels

In [5]:
# Convierte a un dict a los valores de tools que son strings que representan dicts en la columna tools
df_wild["tools"] = df_wild["tools"].apply(ast.literal_eval)

# Extraigo categorias de vulnerabilidades de cada tools elem
df_wild["vulnerability_category_labels"] = df_wild["tools"].apply(extract_labels)

# Obtengo la lista de labels
labels = set()

for category_set in df_wild["vulnerability_category_labels"]:
    if not category_set:
        labels.add("clean")
    else:
        for category in category_set:
            labels.add(category)

print("Labels: ", labels)

Labels:  {'clean', 'other', 'arithmetic', 'time_manipulation', 'front_running', 'unchecked_low_calls', 'access_control', 'reentrancy', 'denial_service'}


In [6]:
label_dict_count = dict.fromkeys(labels, 0)

for category_set in df_wild["vulnerability_category_labels"]:
    if not category_set:
        label_dict_count["clean"] += 1
    else:
        for category in category_set:
            label_dict_count[category] += 1

for label in label_dict_count:
    print(label.ljust(20), "\t\t", label_dict_count[label])

clean                		 2862
other                		 28355
arithmetic           		 37597
time_manipulation    		 4069
front_running        		 8161
unchecked_low_calls  		 14656
access_control       		 3801
reentrancy           		 14747
denial_service       		 12419


In [25]:
FUNCTIONS_PATH = "/kaggle/input/datasets/frmorales/functions-v3"
## FUNCTIONS_PATH = ''

if FUNCTIONS_PATH not in sys.path:
    sys.path.append(FUNCTIONS_PATH)

try:
    from preprocessing import preprocess_source

    print("✅ Módulo cargado correctamente")
except ImportError as e:
    print(f"❌ Error al importar: {e}")
    print("Contenido del dataset:", os.listdir(dataset_path))

from deduplication import deduplicate_df

# from mcd import apply_mcd_filter, T, MU_C, EQUAL_RATE, EQUAL_WEIGHT_CATS
from mcd2 import EQUAL_RATE, EQUAL_WEIGHT_CATS, MU_C, T, apply_mcd_filter
from stats import print_class_balance
from tokenization import filter_by_token_length

✅ Módulo cargado correctamente


In [26]:
#########################################################
### FUNCIÓN: MINIFICACIÓN DE CÓDIGO SOLIDITY ###
#########################################################

"""
TODO
Elimina comentarios para identificar correctamente el último contrato.
Armar otra opción donde no sea necesario eliminar comentarios ya que pueden poseer información valiosa.
"""


def clean_extract_and_minify(code: str) -> str:
    """
    Realiza una limpieza del código fuente para eliminar código de librerias.

    Objetivos principales:
    1. Enfoque de Lógica: Extrae solo el último contrato del archivo (generalmente
       el contrato principal).
    2. Optimización de Tokens: La minificación agresiva permite que una mayor
       cantidad de lógica de control quepa dentro del límite de 512 tokens de CodeBERT.

    Pasos:
    - Regex 1 & 2: Remoción de comentarios multilínea (/* */) y unilínea (//).
    - Regex 3: Identifica todos los 'contract Name' y recorta el string desde el inicio del último.

    Args:
        code (str): Código fuente original de Solidity.

    Returns:
        str: Código minificado.
    """
    if not isinstance(code, str):
        return ""

    code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
    code = re.sub(r"//.*", "", code)

    matches = list(re.finditer(r"\bcontract\s+(\w+)", code))
    if matches:
        last_contract_start = matches[-1].start()
        code = code[last_contract_start:]

    return code.strip()

In [27]:
df_clean = df_wild.copy()

df_clean["source_code"] = df_clean["source_code"].apply(clean_extract_and_minify)

In [28]:
####################################################
### ELIMINACIÓN DE CONTRATOS CON CÓDIGO INVÁLIDO ###
####################################################

n_before = len(df_clean)
df_clean = df_clean[df_clean["source_code"].apply(lambda x: isinstance(x, str) and len(x) > 0)].reset_index(drop=True)
print(f"Contratos con source_code inválido removidos: {n_before - len(df_clean)}")
print(f"Contratos restantes                         : {len(df_clean)}")

Contratos con source_code inválido removidos: 120
Contratos restantes                         : 47331


In [29]:
#####################
### NORMALIZACIÓN ###
#####################

df_clean["source_code"] = df_clean["source_code"].apply(preprocess_source)

In [30]:
###########################################
### ELIMINACIÓN DE CONTRATOS DUPLICADOS ###
###########################################

df_clean = deduplicate_df(df_clean)

Contratos antes de deduplicación : 47331
Duplicados eliminados            : 1967
Contratos restantes              : 45364


In [36]:
##########################################################################
### USO DE MATRIZ DE CAPACIDAD DE DETECCIÓN PARA ELECCIÓN DE CONTRATOS ###
### QUE SUPEREN UN THRESHOLD DE CONFIANZA EN SUS LABELS                ###
##########################################################################

df_verified = df_clean.copy()
print(f"Hiperparámetros: T={T}, μ_c={MU_C}, EQUAL_RATE={EQUAL_RATE}")
print(f"Categorías con peso uniforme: {EQUAL_WEIGHT_CATS}")
print()
df_verified = apply_mcd_filter(df_verified, t=1.2, mu_c=0.3)

Hiperparámetros: T=0.8, μ_c=0.1, EQUAL_RATE=20.0
Categorías con peso uniforme: {'denial_service'}

Hiperparámetros usados: T=1.2, μ_c=0.3
Contratos descartados (ningún label ≥ μ_c=0.3): 14376
Contratos restantes                               : 30988


In [37]:
#################################
### TOKENIZACIÓN DE CONTRATOS ###
#################################

df_tokenized = df_verified.copy()

tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
df_tokenized = filter_by_token_length(df_tokenized, tokenizer, max_tokens=512)

Token indices sequence length is longer than the specified maximum sequence length for this model (2094 > 512). Running this sequence through the model will result in indexing errors


Calculando longitudes de tokenización...
Contratos descartados (> 512 tokens): 13627  (44.0%)
Contratos restantes                          : 17361

count    17361.000000
mean       248.745982
std        126.758106
min          9.000000
25%        124.000000
50%        240.000000
75%        340.000000
max        512.000000


In [38]:
########################################
### SE VUELVEN A ELIMINAR DUPLICADOS ###
########################################

df_final = deduplicate_df(df_tokenized)

print("\n--- Análisis de Impacto del Truncamiento ---")
impacto = len(df_tokenized) - len(df_final)
porcentaje = (impacto / len(df_tokenized)) * 100
print(f"Contratos que se volvieron idénticos post-truncamiento: {impacto} ({porcentaje:.2f}%)")

Contratos antes de deduplicación : 17361
Duplicados eliminados            : 0
Contratos restantes              : 17361

--- Análisis de Impacto del Truncamiento ---
Contratos que se volvieron idénticos post-truncamiento: 0 (0.00%)


In [39]:
#########################################
### ESTADISTICAS DE BALANCEO DE DATOS ###
#########################################

print_class_balance(df_tokenized)

=== Balance de clases (post-cleaning) ===
  arithmetic                 12796  (73.7%)
  reentrancy                  3201  (18.4%)
  other                       1693  (9.8%)
  clean                       1388  (8.0%)
  unchecked_low_calls          137  (0.8%)
  denial_service                84  (0.5%)
  time_manipulation             47  (0.3%)
  access_control                32  (0.2%)

Total contratos  : 17361
Labels / contrato: 1.12 (promedio)


In [40]:
###########################
### SE ALMACENA DATASET ###
###########################

OUTPUT_PATH = "/kaggle/working/dataset2.csv"

df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Total de filas guardadas: {len(df_final)}")

Total de filas guardadas: 17361


In [ ]:
os.environ["HF_TOKEN"] = ""
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRANSFORMERS_CACHE"] = os.path.join(os.getcwd(), "cache")
os.environ["HF_HOME"] = os.path.join(os.getcwd(), "cache")

In [42]:
# Hiperparámetros
MAX_LEN = 512
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-5
WARMUP_RATIO = 0.1

# Configuración uso de GPU T4 X2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | GPUs: {torch.cuda.device_count()}")

Device: cuda | GPUs: 2


In [43]:
TRAIN_PATH = "/kaggle/working/dataset2.csv"
# TRAIN_PATH = "/kaggle/input/datasets/frmorales/train-datasetv-2/discard_plus512_t12_mu03_train.csv"
TEST_PATH = "/kaggle/input/datasets/frmorales/smartbugs-curated/smartbugs_curated_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train: {train_df.shape} | Test: {test_df.shape}")
print(f"Columnas train: {list(train_df.columns)}")
print(f"Columnas test:  {list(test_df.columns)}")

Train: (17361, 11) | Test: (134, 3)
Columnas train: ['address', 'tools', 'lines', 'nb_vulnerabilities', 'source_code', 'source_len', 'vulnerability_category_labels', 'md5_hash', 'scores', 'accepted_labels', 'token_count']
Columnas test:  ['filename', 'vulnerability_category', 'source_code']


In [ ]:
def parse_labels(label_str):
    """Convierte '{label1, label2}' a lista ordenada de labels."""
    label_str = label_str.strip()
    if label_str.startswith("{") and label_str.endswith("}"):
        inner = label_str[1:-1]
        return sorted([item.strip().strip("'\"") for item in inner.split(",") if item.strip()])
    try:
        parsed = ast.literal_eval(label_str)
        if isinstance(parsed, (set, list, tuple)):
            return sorted(str(x) for x in parsed)
        return [str(parsed)]
    except Exception:
        return [label_str.strip()]


train_df["labels_list"] = train_df["accepted_labels"].apply(parse_labels)
test_df["labels_list"] = test_df["vulnerability_category"].apply(lambda x: [x.strip()])

# Binarizar sobre la unión de labels train + test
mlb = MultiLabelBinarizer()
mlb.fit(train_df["labels_list"].tolist() + test_df["labels_list"].tolist())
NUM_LABELS = len(mlb.classes_)

train_encoded = mlb.transform(train_df["labels_list"])
test_encoded = mlb.transform(test_df["labels_list"])

print(f"Clases ({NUM_LABELS}): {list(mlb.classes_)}")
print(
    f"Ejemplo train: {train_df['accepted_labels'].iloc[0]} -> {train_df['labels_list'].iloc[0]} -> {train_encoded[0]}"
)

Clases (9): ['access_control', 'arithmetic', 'clean', 'denial_service', 'front_running', 'other', 'reentrancy', 'time_manipulation', 'unchecked_low_calls']
Ejemplo train: {'clean'} -> ['clean'] -> [0 0 1 0 0 0 0 0 0]


In [45]:
# Carga del tokenizador y modelo ahora que tenemos la cantidad de labels

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base", num_labels=NUM_LABELS, problem_type="multi_label_classification"
)

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [46]:
def tokenize_function(examples):
    return tokenizer(
        examples["source_code"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )


# Crear datasets de HF con source_code y labels (float para BCEWithLogitsLoss)
train_hf = HFDataset.from_dict(
    {
        "source_code": train_df["source_code"].tolist(),
        "labels": [row.tolist() for row in train_encoded.astype(np.float32)],
    }
)

test_hf = HFDataset.from_dict(
    {
        "source_code": test_df["source_code"].tolist(),
        "labels": [row.tolist() for row in test_encoded.astype(np.float32)],
    }
)

# Tokenizar
train_hf = train_hf.map(tokenize_function, batched=True)
test_hf = test_hf.map(tokenize_function, batched=True)

# Remover columna de texto crudo (el Trainer solo necesita input_ids, attention_mask, labels)
train_hf = train_hf.remove_columns(["source_code"])
test_hf = test_hf.remove_columns(["source_code"])

train_hf.set_format("torch")
test_hf.set_format("torch")

print(f"Train: {len(train_hf)} | Test: {len(test_hf)}")
print(f"Features: {list(train_hf.features.keys())}")
print(f"Labels shape: {train_hf[0]['labels'].shape}")

Map:   0%|          | 0/17361 [00:00<?, ? examples/s]

Map:   0%|          | 0/134 [00:00<?, ? examples/s]

Train: 17361 | Test: 134
Features: ['labels', 'input_ids', 'attention_mask']
Labels shape: torch.Size([9])


In [47]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()

    pred_flat = np.argmax(probs, axis=1)
    true_flat = np.argmax(labels, axis=1)

    return {
        "f1_macro": f1_score(true_flat, pred_flat, average="macro", zero_division=0),
        "f1_micro": f1_score(true_flat, pred_flat, average="micro", zero_division=0),
        "recall": recall_score(true_flat, pred_flat, average="macro", zero_division=0),
        "accuracy": accuracy_score(true_flat, pred_flat),
    }

In [48]:
######
###### weighted trainer
######

# Calcular pesos por clase sobre el set de train. Peso ponderado positivos vs negativos (no_es_la_label / es_la_label)
LR = 2e-5
EPOCHS = 5

class_freqs = train_encoded.sum(axis=0)
total_samples = len(train_encoded)
pos_weights = (total_samples - class_freqs) / np.maximum(class_freqs, 1)
pos_weights_smoothed = np.sqrt(pos_weights)
pos_weights_tensor = torch.tensor(pos_weights_smoothed, dtype=torch.float).to(DEVICE)

print("Pesos por clase:")
for cls, w in zip(mlb.classes_, pos_weights_smoothed):
    print(f"  {cls}: {w:.2f}")


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").to(self.args.device)
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


# Training arguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/codebert_smartbugs",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=int((len(train_hf) / BATCH_SIZE) * EPOCHS * WARMUP_RATIO),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,  # P100/T4 soportan fp16
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    compute_metrics=compute_metrics,
)

trainer.train()

Pesos por clase:
  access_control: 23.27
  arithmetic: 0.60
  clean: 3.39
  denial_service: 14.34
  front_running: 131.76
  other: 3.04
  reentrancy: 2.10
  time_manipulation: 19.19
  unchecked_low_calls: 11.21


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Recall,Accuracy
1,0.299121,3.469587,0.057281,0.111940,0.084211,0.111940
2,0.203067,3.129387,0.157470,0.253731,0.165550,0.253731
3,0.146896,3.610784,0.194054,0.291045,0.176234,0.291045
4,0.120591,3.332602,0.144357,0.276119,0.171841,0.276119


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
results = trainer.evaluate()

print("\n" + "=" * 50)
print("TEST RESULTS")
print("=" * 50)
print(f"  F1 Macro : {results['eval_f1_macro']:.4f}")
print(f"  F1 Micro : {results['eval_f1_micro']:.4f}")
print(f"  Recall   : {results['eval_recall']:.4f}")
print(f"  Accuracy : {results['eval_accuracy']:.4f}")
print("=" * 50)

# Guardar modelo y tokenizer
save_path = "/kaggle/working/codebert_smartbugs_weighted"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"\nModelo guardado en {save_path}")

In [ ]:
!zip -r output.zip /kaggle/working/codebert_smartbugs

In [ ]:
def get_predictions(trainer, dataset, mlb, logits_raw=None):
    if logits_raw is None:
        output = trainer.predict(dataset)
        logits_raw = output.predictions
        labels = output.label_ids
    else:
        labels = np.array([dataset[i]["labels"].numpy() for i in range(len(dataset))])

    probs = torch.sigmoid(torch.tensor(logits_raw)).numpy()

    # Para test single-label: tomar la clase con mayor probabilidad
    pred_flat = np.argmax(probs, axis=1)
    true_flat = np.argmax(labels, axis=1)

    pred_names = [mlb.classes_[i] for i in pred_flat]
    true_names = [mlb.classes_[i] for i in true_flat]

    return {
        "pred_flat": pred_flat,
        "true_flat": true_flat,
        "pred_names": pred_names,
        "true_names": true_names,
        "probs": probs,
        "logits": logits_raw,
        "labels": labels,
        "classes": list(mlb.classes_),
    }


def plot_confusion_matrix(results, figsize=(10, 8)):
    """Muestra la matriz de confusión con porcentajes."""
    classes = results["classes"]
    cm = confusion_matrix(results["true_flat"], results["pred_flat"])

    # Normalizar por fila (recall por clase)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    fig, axes = plt.subplots(1, 2, figsize=(figsize[0] * 2, figsize[1]))

    # Absoluta
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, ax=axes[0])
    axes[0].set_xlabel("Predicho")
    axes[0].set_ylabel("Real")
    axes[0].set_title("Matriz de Confusión (absoluta)")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].tick_params(axis="y", rotation=0)

    # Normalizada
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
        ax=axes[1],
    )
    axes[1].set_xlabel("Predicho")
    axes[1].set_ylabel("Real")
    axes[1].set_title("Matriz de Confusión (normalizada)")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].tick_params(axis="y", rotation=0)

    plt.tight_layout()
    plt.show()


def print_classification_report(results):
    """Imprime el reporte de clasificación por clase."""
    print(
        classification_report(
            results["true_names"],
            results["pred_names"],
            zero_division=0,
        )
    )


def show_samples(results, test_df, tokenizer, n=20, filter_wrong=False, filter_class=None, save_path=None):
    indices = list(range(len(results["true_names"])))

    if filter_wrong:
        indices = [i for i in indices if results["true_names"][i] != results["pred_names"][i]]

    if filter_class:
        indices = [i for i in indices if results["true_names"][i] == filter_class]

    indices = indices[:n]

    lines = []
    header = (
        f"Mostrando {len(indices)} muestras"
        + (" (solo errores)" if filter_wrong else "")
        + (f" (clase: {filter_class})" if filter_class else "")
    )
    lines.append(header)
    lines.append("=" * 100)

    for i in indices:
        code_full = str(test_df["source_code"].iloc[i])
        num_tokens = len(tokenizer.encode(code_full))
        truncado = " ⚠ TRUNCADO" if num_tokens > MAX_LEN else ""
        correct = "✓" if results["true_names"][i] == results["pred_names"][i] else "✗"

        top3_idx = np.argsort(results["probs"][i])[::-1][:3]
        top3 = [(results["classes"][j], results["probs"][i][j]) for j in top3_idx]
        top3_str = " | ".join([f"{name}: {prob:.3f}" for name, prob in top3])

        lines.append(f"[{correct}] Sample {i}")
        lines.append(f"  Real:      {results['true_names'][i]}")
        lines.append(f"  Predicho:  {results['pred_names'][i]}")
        lines.append(f"  Tokens:    {num_tokens} / {MAX_LEN}{truncado}")
        lines.append(f"  Top 3:     {top3_str}")
        lines.append("  Código completo:")
        lines.append("-" * 50)
        lines.append(code_full)
        lines.append("=" * 100)

    output = "\n".join(lines)
    print(output)

    if save_path:
        with open(save_path, "w") as f:
            f.write(output)
        print(f"\nGuardado en {save_path}")


def show_class_distribution(results):
    """Muestra distribución de clases reales vs predichas."""
    classes = results["classes"]
    true_counts = np.bincount(results["true_flat"], minlength=len(classes))
    pred_counts = np.bincount(results["pred_flat"], minlength=len(classes))

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(classes))
    width = 0.35

    ax.bar(x - width / 2, true_counts, width, label="Real", color="steelblue")
    ax.bar(x + width / 2, pred_counts, width, label="Predicho", color="salmon")
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_ylabel("Cantidad")
    ax.set_title("Distribución de clases: Real vs Predicho")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Obtener predicciones
res = get_predictions(trainer, test_hf, mlb)

# Matriz de confusión
plot_confusion_matrix(res)

# Reporte por clase
print_classification_report(res)

# Distribución real vs predicho
show_class_distribution(res)

# Ver errores de una clase específica
show_samples(
    res,
    test_df,
    tokenizer,
    n=30,
    filter_wrong=True,
    save_path="/kaggle/working/errores_detalle.txt",
)

In [ ]:
# Distribución de clases en train
print("TRAIN:")
print(train_df["labels_list"].explode().value_counts())
print()

# Distribución de clases en test
print("TEST:")
print(test_df["labels_list"].explode().value_counts())

In [ ]:
def generate_report(
    trainer,
    test_df,
    test_hf,
    mlb,
    tokenizer,
    run_name="run",
    hyperparams=None,
    save_dir="/kaggle/working",
):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = f"{save_dir}/{run_name}_{timestamp}"

    res = get_predictions(trainer, test_hf, mlb)
    classes = res["classes"]  # <-- definir acá

    report_dict = classification_report(
        res["true_names"],
        res["pred_names"],
        zero_division=0,
        output_dict=True,
    )

    lines = []
    lines.append("=" * 80)
    lines.append(f"REPORTE DE ENTRENAMIENTO: {run_name}")
    lines.append(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append("=" * 80)

    lines.append("\n--- HIPERPARÁMETROS ---")
    if hyperparams:
        for k, v in hyperparams.items():
            lines.append(f"  {k}: {v}")
    else:
        lines.append("  (no especificados)")

    lines.append("\n--- MÉTRICAS GENERALES ---")
    lines.append(f"  F1 Macro : {report_dict['macro avg']['f1-score']:.4f}")
    lines.append(f"  F1 Micro : {report_dict.get('weighted avg', {}).get('f1-score', 0):.4f}")
    lines.append(f"  Recall   : {report_dict['macro avg']['recall']:.4f}")
    lines.append(f"  Accuracy : {report_dict['accuracy']:.4f}")

    lines.append("\n--- MÉTRICAS POR CLASE ---")
    lines.append(f"  {'Clase':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    lines.append("  " + "-" * 65)
    for cls in mlb.classes_:
        if cls in report_dict:
            r = report_dict[cls]
            lines.append(
                f"  {cls:<25} {r['precision']:>10.4f} {r['recall']:>10.4f} {r['f1-score']:>10.4f} {r['support']:>10.0f}"
            )

    lines.append("\n--- DISTRIBUCIÓN TRAIN ---")
    train_counts = train_df["labels_list"].explode().value_counts()
    for cls, count in train_counts.items():
        lines.append(f"  {cls:<25} {count}")

    lines.append("\n--- DISTRIBUCIÓN TEST ---")
    test_counts = test_df["labels_list"].explode().value_counts()
    for cls, count in test_counts.items():
        lines.append(f"  {cls:<25} {count}")

    lines.append("\n--- MATRIZ DE CONFUSIÓN ---")
    cm = confusion_matrix(res["true_flat"], res["pred_flat"], labels=list(range(len(classes))))
    header = f"  {'':>22}" + "".join(f"{c[:12]:>13}" for c in classes)
    lines.append(header)
    for i, cls in enumerate(classes):
        row = f"  {cls:>22}" + "".join(f"{cm[i][j]:>13}" for j in range(len(classes)))
        lines.append(row)

    lines.append("\n--- HISTORIAL DE ENTRENAMIENTO ---")
    if trainer.state.log_history:
        for entry in trainer.state.log_history:
            if "loss" in entry or "eval_loss" in entry:
                lines.append(f"  {entry}")

    lines.append("\n\n" + "=" * 80)
    lines.append("DETALLE DE TODAS LAS PREDICCIONES (TEST)")
    lines.append("=" * 80)

    wrong_indices = [i for i in range(len(res["true_names"])) if res["true_names"][i] != res["pred_names"][i]]
    correct_indices = [i for i in range(len(res["true_names"])) if res["true_names"][i] == res["pred_names"][i]]

    lines.append(f"\nTotal: {len(res['true_names'])} samples")
    lines.append(f"  Correctos: {len(correct_indices)} ({len(correct_indices) / len(res['true_names']) * 100:.1f}%)")
    lines.append(f"  Errores:   {len(wrong_indices)} ({len(wrong_indices) / len(res['true_names']) * 100:.1f}%)")

    for i in range(len(res["true_names"])):
        code_full = str(test_df["source_code"].iloc[i])
        num_tokens = len(tokenizer.encode(code_full))

        top3_idx = np.argsort(res["probs"][i])[::-1][:3]
        top3 = [(classes[j], res["probs"][i][j]) for j in top3_idx]
        top3_str = " | ".join([f"{name}: {prob:.3f}" for name, prob in top3])

        is_correct = res["true_names"][i] == res["pred_names"][i]
        mark = "✓" if is_correct else "✗"

        lines.append(f"\n[{mark}] Sample {i}")
        lines.append(f"  Real:      {res['true_names'][i]}")
        lines.append(f"  Predicho:  {res['pred_names'][i]}")
        lines.append(f"  Tokens:    {num_tokens}")
        lines.append(f"  Top 3:     {top3_str}")
        lines.append("  Código:")
        lines.append("-" * 50)
        lines.append(code_full)
        lines.append("-" * 50)

    report_path = f"{prefix}_report.txt"
    with open(report_path, "w") as f:
        f.write("\n".join(lines))

    json_path = f"{prefix}_metrics.json"
    json_data = {
        "run_name": run_name,
        "timestamp": timestamp,
        "hyperparams": hyperparams or {},
        "metrics": {
            "f1_macro": report_dict["macro avg"]["f1-score"],
            "f1_micro": report_dict.get("weighted avg", {}).get("f1-score", 0),
            "recall": report_dict["macro avg"]["recall"],
            "accuracy": report_dict["accuracy"],
        },
        "per_class": {cls: report_dict.get(cls, {}) for cls in mlb.classes_},
    }
    with open(json_path, "w") as f:
        json.dump(json_data, f, indent=2)

    cm_path = f"{prefix}_confusion_matrix.png"
    cm = confusion_matrix(res["true_flat"], res["pred_flat"], labels=list(range(len(classes))))
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, ax=axes[0])
    axes[0].set_xlabel("Predicho")
    axes[0].set_ylabel("Real")
    axes[0].set_title("Matriz de Confusión (absoluta)")
    axes[0].tick_params(axis="x", rotation=45)

    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
        ax=axes[1],
    )
    axes[1].set_xlabel("Predicho")
    axes[1].set_ylabel("Real")
    axes[1].set_title("Matriz de Confusión (normalizada)")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("\nArchivos guardados:")
    print(f"  Reporte:  {report_path}")
    print(f"  Métricas: {json_path}")
    print(f"  Matriz:   {cm_path}")

    return res

In [ ]:
res = generate_report(
    trainer,
    test_df,
    test_hf,
    mlb,
    tokenizer,
    run_name="v2_con_pesos_lr0_0001",
    hyperparams={
        "model": "microsoft/codebert-base",
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "warmup_steps": 100,
        "weight_decay": 0.01,
        "pos_weight": "ninguno",
        "fp16": True,
        "notas": "Sin weighted trainer, menos lr",
    },
)